In [15]:
!pip install ultralytics
!pip install tqdm
!pip install scikit-learn

In [53]:
import os
import shutil
from sklearn.model_selection import train_test_split

# =========================================================
# DATASET 1
# =========================================================
DATASET1_IMAGES = "/mnt/DATA/FRAMES3/frames3"
DATASET1_LABELS = "/mnt/DATA/FRAMES3/labels"

# =========================================================
# DATASET 2
# =========================================================
DATASET2_IMAGES = "/mnt/DATA/ir_dataset/images"
DATASET2_LABELS = "/mnt/DATA/ir_dataset/labels"

# =========================================================
# DATASET 3 (ALREADY SPLIT)
# =========================================================
DATASET3_PATH = "/mnt/DATA/SPLAB/Downloads/Dataset_Mendeley_data/Labelled_UAV_Images/Labelled EO_IR Dataset/UAV_ir_Dataset"

# =========================================================
# FINAL OUTPUT DATASET
# =========================================================
FINAL_DATASET = "/mnt/DATA/final_training_dataset"

# Create folders
for split in ["train", "val"]:
    os.makedirs(os.path.join(FINAL_DATASET, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(FINAL_DATASET, split, "labels"), exist_ok=True)

# =========================================================
# FUNCTION TO COPY FILES
# =========================================================
def copy_files(file_list, image_folder, label_folder, split_name, prefix):
    for img_name in file_list:
        # -------------------------------
        # NEW UNIQUE FILENAMES
        # -------------------------------
        new_img_name = prefix + "_" + img_name
        label_name = os.path.splitext(img_name)[0] + ".txt"
        new_label_name = prefix + "_" + label_name

        # -------------------------------
        # SOURCE PATHS
        # -------------------------------
        src_img = os.path.join(image_folder, img_name)
        src_label = os.path.join(label_folder, label_name)

        # -------------------------------
        # DESTINATION PATHS
        # -------------------------------
        dst_img = os.path.join(FINAL_DATASET, split_name, "images", new_img_name)
        dst_label = os.path.join(FINAL_DATASET, split_name, "labels", new_label_name)

        # -------------------------------
        # COPY
        # -------------------------------
        if os.path.exists(src_img) and os.path.exists(src_label):
            shutil.copy(src_img, dst_img)
            shutil.copy(src_label, dst_label)

# =========================================================
# DATASET 1 SPLIT (80 TRAIN / 20 VAL)
# =========================================================
print("\nProcessing Dataset 1...")
dataset1_images = [
    f for f in os.listdir(DATASET1_IMAGES)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

train1, val1 = train_test_split(dataset1_images, test_size=0.20, random_state=42)

copy_files(train1, DATASET1_IMAGES, DATASET1_LABELS, "train", "d1")
copy_files(val1, DATASET1_IMAGES, DATASET1_LABELS, "val", "d1")

print(f"Dataset1 Train: {len(train1)}")
print(f"Dataset1 Val : {len(val1)}")

# =========================================================
# DATASET 2 SPLIT (80 TRAIN / 20 VAL)
# =========================================================
print("\nProcessing Dataset 2...")
dataset2_images = [
    f for f in os.listdir(DATASET2_IMAGES)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

train2, val2 = train_test_split(dataset2_images, test_size=0.20, random_state=42)

copy_files(train2, DATASET2_IMAGES, DATASET2_LABELS, "train", "d2")
copy_files(val2, DATASET2_IMAGES, DATASET2_LABELS, "val", "d2")

print(f"Dataset2 Train: {len(train2)}")
print(f"Dataset2 Val : {len(val2)}")

# =========================================================
# COPY DATASET 3 TRAIN + VAL
# =========================================================
print("\nCopying Dataset 3...")
for split in ["train", "val"]:
    image_folder = os.path.join(DATASET3_PATH, split, "images")
    label_folder = os.path.join(DATASET3_PATH, split, "labels")

    if os.path.exists(image_folder):
        image_files = [
            f for f in os.listdir(image_folder)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]
        copy_files(image_files, image_folder, label_folder, split, "d3")
        print(f"Dataset3 {split}: {len(image_files)}")

# =========================================================
# FINAL SUMMARY
# =========================================================
print("\n===================================")
print("FINAL DATASET CREATED")
print("===================================")

for split in ["train", "val"]:
    image_count = len(os.listdir(os.path.join(FINAL_DATASET, split, "images")))
    label_count = len(os.listdir(os.path.join(FINAL_DATASET, split, "labels")))

    print(f"\n{split.upper()}")
    print(f"Images : {image_count}")
    print(f"Labels : {label_count}")

print(f"\nSaved at:\n{FINAL_DATASET}")


Processing Dataset 1...
Dataset1 Train: 326
Dataset1 Val : 82

Processing Dataset 2...
Dataset2 Train: 456
Dataset2 Val : 114

Copying Dataset 3...

FINAL DATASET CREATED

TRAIN
Images : 5978
Labels : 5978

VAL
Images : 1105
Labels : 1105

Saved at:
/mnt/DATA/final_training_dataset


In [54]:
yaml_content = """
path: /mnt/DATA/final_training_dataset

train: train/images
val: val/images

nc: 1

names:
  0: drone
"""

with open("/mnt/DATA/final_ir_dataset.yaml", "w") as f:
    f.write(yaml_content)

print("✅ YAML file created!")
print("/mnt/DATA/final_ir_dataset.yaml")

✅ YAML file created!
/mnt/DATA/final_ir_dataset.yaml


In [55]:
from ultralytics import YOLO

# =========================================
# LOAD MODEL
# =========================================
model = YOLO("yolo11n.pt")
# or use:
# model = YOLO("yolov8n.pt")

# =========================================
# TRAIN
# =========================================
results = model.train(
    data="/mnt/DATA/final_ir_dataset.yaml",
    
    epochs=50,
    imgsz=1024.,
    batch=16,

    device=0,      # GPU
    workers=4,

    project="/mnt/DATA/yolo_ir_results",
    name="drone_detection",

    patience=20,
    save=True,

    pretrained=True
)

print("✅ Training Complete!")

New https://pypi.org/project/ultralytics/8.4.48 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.47 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A4000, 15985MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/DATA/final_ir_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0,

In [56]:
from ultralytics import YOLO

model = YOLO("/mnt/DATA/yolo_ir_results/drone_detection-5/weights/best.pt")

metrics = model.val()

print(metrics)

Ultralytics 8.4.47 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A4000, 15985MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6023.2±1783.8 MB/s, size: 95.9 KB)
val: Scanning /mnt/DATA/final_training_dataset/val/labels.cache... 1105 images, 216 backgrounds, 32 corrupt: 100% ━━━━━━━━━━━━ 1105/1105 331.1Mit/s 0.0s
val: /mnt/DATA/final_training_dataset/val/images/d1_frame_00000.jpg: ignoring corrupt image/label: labels require 5 columns, 6 columns detected
val: /mnt/DATA/final_training_dataset/val/images/d1_frame_00002.jpg: ignoring corrupt image/label: labels require 5 columns, 6 columns detected
val: /mnt/DATA/final_training_dataset/val/images/d1_frame_00011.jpg: ignoring corrupt image/label: labels require 5 columns, 6 columns detected
val: /mnt/DATA/final_training_dataset/val/images/d1_frame_00026.jpg: ignoring corrupt image/label: labels require 5 columns, 6 columns detected
val: /mnt/DAT

In [40]:
import os
from tqdm import tqdm

# ==========================================
# LABEL FOLDER
# ==========================================
label_folders = [
    "/mnt/DATA/final_training_dataset/train/labels",
    "/mnt/DATA/final_training_dataset/val/labels"
]

# ==========================================
# FIX LABELS
# ==========================================
for folder in label_folders:

    label_files = [
        f for f in os.listdir(folder)
        if f.endswith(".txt")
    ]

    print(f"\nProcessing: {folder}")

    for file in tqdm(label_files):

        path = os.path.join(folder, file)

        new_lines = []

        with open(path, "r") as f:
            lines = f.readlines()

        for line in lines:

            parts = line.strip().split()

            # Keep only first 5 columns
            if len(parts) >= 5:
                fixed = parts[:5]
                new_lines.append(" ".join(fixed))

        # Rewrite file
        with open(path, "w") as f:
            f.write("\n".join(new_lines))

print("\n✅ All labels fixed to YOLO format")


Processing: /mnt/DATA/final_training_dataset/train/labels


100%|██████████| 5857/5857 [00:00<00:00, 15556.10it/s]



Processing: /mnt/DATA/final_training_dataset/val/labels


100%|██████████| 1073/1073 [00:00<00:00, 17225.76it/s]


✅ All labels fixed to YOLO format


In [57]:
from ultralytics import YOLO

# ==========================================
# LOAD TRAINED MODEL
# ==========================================
model = YOLO(
    "/mnt/DATA/yolo_ir_results/drone_detection-5/weights/best.pt"
)

# ==========================================
# VALIDATE MODEL
# ==========================================
metrics = model.val(
    data="/mnt/DATA/final_ir_dataset.yaml",
    imgsz=640,
    conf=0.30
)

# ==========================================
# PRINT METRICS
# ==========================================
print("\n==============================")
print("MODEL METRICS")
print("==============================")

print(f"mAP50       : {metrics.box.map50:.4f}")
print(f"mAP50-95    : {metrics.box.map:.4f}")

print(f"Precision   : {metrics.box.mp:.4f}")
print(f"Recall      : {metrics.box.mr:.4f}")

print("\n✅ Validation Complete")

Ultralytics 8.4.47 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A4000, 15985MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4911.3±1862.3 MB/s, size: 117.4 KB)
val: Scanning /mnt/DATA/final_training_dataset/val/labels.cache... 1105 images, 216 backgrounds, 32 corrupt: 100% ━━━━━━━━━━━━ 1105/1105 386.2Mit/s 0.0s
val: /mnt/DATA/final_training_dataset/val/images/d1_frame_00000.jpg: ignoring corrupt image/label: labels require 5 columns, 6 columns detected
val: /mnt/DATA/final_training_dataset/val/images/d1_frame_00002.jpg: ignoring corrupt image/label: labels require 5 columns, 6 columns detected
val: /mnt/DATA/final_training_dataset/val/images/d1_frame_00011.jpg: ignoring corrupt image/label: labels require 5 columns, 6 columns detected
val: /mnt/DATA/final_training_dataset/val/images/d1_frame_00026.jpg: ignoring corrupt image/label: labels require 5 columns, 6 columns detected
val: /mnt/DA

In [58]:
from ultralytics import YOLO

# ==========================================
# LOAD TRAINED MODEL
# ==========================================
model = YOLO(
    "/mnt/DATA/yolo_ir_results/drone_detection-5/weights/best.pt"
)

# ==========================================
# TEST ON YOUR THERMAL FRAMES
# ==========================================
results = model.predict(

    # Folder containing images
    source="/mnt/DATA/frames",

    # Save detection images
    save=True,

    # Confidence threshold
    conf=0.25,

    # Image size
    imgsz=640,

    # Show labels + confidence
    show_labels=True,
    show_conf=True
)

print("✅ Testing Complete")


image 1/405 /mnt/DATA/frames/frame_00000.jpg: 512x640 1 drone, 6.5ms
image 2/405 /mnt/DATA/frames/frame_00001.jpg: 512x640 1 drone, 5.0ms
image 3/405 /mnt/DATA/frames/frame_00002.jpg: 512x640 1 drone, 5.8ms
image 4/405 /mnt/DATA/frames/frame_00003.jpg: 512x640 1 drone, 5.3ms
image 5/405 /mnt/DATA/frames/frame_00004.jpg: 512x640 1 drone, 5.1ms
image 6/405 /mnt/DATA/frames/frame_00005.jpg: 512x640 1 drone, 6.1ms
image 7/405 /mnt/DATA/frames/frame_00006.jpg: 512x640 1 drone, 5.3ms
image 8/405 /mnt/DATA/frames/frame_00007.jpg: 512x640 1 drone, 6.2ms
image 9/405 /mnt/DATA/frames/frame_00008.jpg: 512x640 1 drone, 5.6ms
image 10/405 /mnt/DATA/frames/frame_00009.jpg: 512x640 1 drone, 5.1ms
image 11/405 /mnt/DATA/frames/frame_00010.jpg: 512x640 1 drone, 6.6ms
image 12/405 /mnt/DATA/frames/frame_00011.jpg: 512x640 1 drone, 5.0ms
image 13/405 /mnt/DATA/frames/frame_00012.jpg: 512x640 1 drone, 5.4ms
image 14/405 /mnt/DATA/frames/frame_00013.jpg: 512x640 1 drone, 5.6ms
image 15/405 /mnt/DATA/frame

In [59]:
from ultralytics import YOLO

# ==========================================
# LOAD TRAINED MODEL
# ==========================================
model = YOLO(
    "/mnt/DATA/yolo_ir_results/drone_detection-5/weights/best.pt"
)

# ==========================================
# TEST ON YOUR THERMAL FRAMES
# ==========================================
results = model.predict(

    # Folder containing images
    source="/mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2",

    # Save detection images
    save=True,

    # Confidence threshold
    conf=0.25,

    # Image size
    imgsz=640,

    # Show labels + confidence
    show_labels=True,
    show_conf=True
)

print("✅ Testing Complete")


image 1/408 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2/frame_00000.jpg: 512x640 1 drone, 6.6ms
image 2/408 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2/frame_00001.jpg: 512x640 1 drone, 5.5ms
image 3/408 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2/frame_00002.jpg: 512x640 1 drone, 5.7ms
image 4/408 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2/frame_00003.jpg: 512x640 1 drone, 6.5ms
image 5/408 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2/frame_00004.jpg: 512x640 1 drone, 5.2ms
image 6/408 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2/frame_00005.jpg: 512x640 1 drone, 5.8ms
image 7/408 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2/frame_00006.jpg: 512x640 1 drone, 6.2ms
image 8/408 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2/frame_00007.jpg: 512x640 1 drone, 5.4ms
image 9/408 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2/frame_00008.jpg: 512x640 1 drone, 6.4ms
image 10/408 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames2/frame_00009.jpg: 5

In [61]:
from ultralytics import YOLO

# ==========================================
# LOAD TRAINED MODEL
# ==========================================
model = YOLO(
    "/mnt/DATA/yolo_ir_results/drone_detection-5/weights/best.pt"
)

# ==========================================
# TEST ON YOUR THERMAL FRAMES
# ==========================================
results = model.predict(

    # Folder containing images
    source="/mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4",

    # Save detection images
    save=True,

    # Confidence threshold
    conf=0.25,

    # Image size
    imgsz=640,

    # Show labels + confidence
    show_labels=True,
    show_conf=True
)

print("✅ Testing Complete")


image 1/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00000.jpg: 512x640 1 drone, 6.3ms
image 2/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00001.jpg: 512x640 1 drone, 4.8ms
image 3/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00002.jpg: 512x640 1 drone, 5.5ms
image 4/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00003.jpg: 512x640 1 drone, 4.8ms
image 5/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00004.jpg: 512x640 1 drone, 5.3ms
image 6/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00005.jpg: 512x640 1 drone, 5.3ms
image 7/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00006.jpg: 512x640 1 drone, 5.5ms
image 8/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00007.jpg: 512x640 1 drone, 6.0ms
image 9/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00008.jpg: 512x640 1 drone, 5.0ms
image 10/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00009.jpg: 5

In [33]:
from ultralytics import YOLO

# ==========================================
# LOAD TRAINED MODEL
# ==========================================
model = YOLO(
    "/mnt/DATA/yolo_ir_results/drone_detection-4/weights/best.pt"
)

# ==========================================
# TEST ON YOUR THERMAL FRAMES
# ==========================================
results = model.predict(

    # Folder containing images
    source="/mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4",

    # Save detection images
    save=True,

    # Confidence threshold
    conf=0.45,

    # Image size
    imgsz=1920,

    # Show labels + confidence
    show_labels=True,
    show_conf=True
)

print("✅ Testing Complete")


image 1/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00000.jpg: 1536x1920 1 drone, 13.3ms
image 2/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00001.jpg: 1536x1920 1 drone, 12.9ms
image 3/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00002.jpg: 1536x1920 2 drones, 13.0ms
image 4/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00003.jpg: 1536x1920 3 drones, 12.9ms
image 5/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00004.jpg: 1536x1920 3 drones, 13.0ms
image 6/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00005.jpg: 1536x1920 3 drones, 12.9ms
image 7/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00006.jpg: 1536x1920 3 drones, 12.4ms
image 8/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00007.jpg: 1536x1920 4 drones, 12.5ms
image 9/405 /mnt/DATA/SPLAB/drone/thermal/label.ipynb/frames4/frame_00008.jpg: 1536x1920 3 drones, 12.4ms
image 10/405 /mnt/DATA/SPLAB/drone/thermal/labe

In [5]:
import os
import shutil
from tqdm import tqdm

# --- CORRECTED PATHS ---
# Removed 'label.ipynb' from the path
SOURCE_IMG_DIR = "/mnt/DATA/SPLAB/drone/thermal/label.ipynb/runs/detect/predict-7"
SOURCE_LBL_DIR = os.path.join(SOURCE_IMG_DIR, "labels")
DEST_ROOT = "/mnt/DATA/ir_dataset"

# Make sure destination folders exist
os.makedirs(os.path.join(DEST_ROOT, "images"), exist_ok=True)
os.makedirs(os.path.join(DEST_ROOT, "labels"), exist_ok=True)

# Verify source exists before running
if os.path.exists(SOURCE_IMG_DIR):
    all_files = [f for f in os.listdir(SOURCE_IMG_DIR) if f.startswith("frame_") and f.endswith(".jpg")]
    print(f"Adding {len(all_files)} new frames to {DEST_ROOT}...")

    added_count = 0
    for img_name in tqdm(all_files):
        lbl_name = img_name.replace(".jpg", ".txt")
        
        src_img = os.path.join(SOURCE_IMG_DIR, img_name)
        src_lbl = os.path.join(SOURCE_LBL_DIR, lbl_name)
        
        dst_img = os.path.join(DEST_ROOT, "images", img_name)
        dst_lbl = os.path.join(DEST_ROOT, "labels", lbl_name)
        
        # Copy Image
        shutil.copy(src_img, dst_img)
        
        # Copy Label (create empty if no drone was detected)
        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, dst_lbl)
        else:
            if not os.path.exists(dst_lbl):
                open(dst_lbl, 'a').close()
        added_count += 1
    
    print(f"\n✅ Done! Added {added_count} frames.")
    print(f"Total images in dataset: {len(os.listdir(os.path.join(DEST_ROOT, 'images')))}")
else:
    print(f"❌ Error: Directory not found. Please check: {SOURCE_IMG_DIR}")

Adding 408 new frames to /mnt/DATA/ir_dataset...


100%|██████████| 408/408 [00:00<00:00, 11468.60it/s]


✅ Done! Added 408 frames.
Total images in dataset: 978


In [12]:
import os
from tqdm import tqdm

# --- CONFIGURATION ---
DEST_ROOT = "/mnt/DATA/ir_dataset"
IMG_DIR = os.path.join(DEST_ROOT, "images")
LBL_DIR = os.path.join(DEST_ROOT, "labels")

# 1. Identify only the "frame_" files
# We check images first to get the base names
files_to_remove = [f for f in os.listdir(IMG_DIR) if f.startswith("frame_")]

if not files_to_remove:
    print("✨ No 'frame_' files found. The dataset is already clean!")
else:
    print(f"🗑️ Removing {len(files_to_remove)} added frames and their labels...")

    removed_count = 0
    for img_name in tqdm(files_to_remove):
        lbl_name = img_name.replace(".jpg", ".txt")
        
        img_path = os.path.join(IMG_DIR, img_name)
        lbl_path = os.path.join(LBL_DIR, lbl_name)
        
        # Delete Image
        if os.path.exists(img_path):
            os.remove(img_path)
            
        # Delete Label
        if os.path.exists(lbl_path):
            os.remove(lbl_path)
            
        removed_count += 1

    print(f"\n✅ Clean-up complete! Removed {removed_count} pairs.")
    print(f"Remaining images in dataset: {len(os.listdir(IMG_DIR))}")

🗑️ Removing 408 added frames and their labels...


100%|██████████| 408/408 [00:00<00:00, 36017.76it/s]


✅ Clean-up complete! Removed 408 pairs.
Remaining images in dataset: 614


In [14]:
import os
import shutil
from tqdm import tqdm

# --- SOURCE PATHS (Based on your screenshot) ---
# Note: In Linux, folders with spaces or dots in the name need careful pathing
SOURCE_ROOT = "/mnt/DATA/SPLAB/drone/thermal/label.ipynb/FRAMES3"
SOURCE_IMG_DIR = os.path.join(SOURCE_ROOT, "frames3")
SOURCE_LBL_DIR = os.path.join(SOURCE_ROOT, "labels")

# --- DESTINATION PATHS ---
DEST_ROOT = "/mnt/DATA/ir_dataset"
DEST_IMG_DIR = os.path.join(DEST_ROOT, "images")
DEST_LBL_DIR = os.path.join(DEST_ROOT, "labels")

# Check if source exists
if os.path.exists(SOURCE_IMG_DIR):
    all_images = [f for f in os.listdir(SOURCE_IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"📂 Found {len(all_images)} images. Adding them with prefix 'f3_'...")

    added_count = 0
    for img_name in tqdm(all_images):
        # Unique naming to prevent overwriting your existing 0000... files
        base_name = os.path.splitext(img_name)[0]
        new_img_name = f"f3_{base_name}.jpg"
        new_lbl_name = f"f3_{base_name}.txt"
        
        src_img = os.path.join(SOURCE_IMG_DIR, img_name)
        src_lbl = os.path.join(SOURCE_LBL_DIR, f"{base_name}.txt")
        
        dst_img = os.path.join(DEST_IMG_DIR, new_img_name)
        dst_lbl = os.path.join(DEST_LBL_DIR, new_lbl_name)
        
        # Copy the image
        shutil.copy(src_img, dst_img)
        
        # Copy the label if it exists, otherwise create a blank one
        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, dst_lbl)
        else:
            if not os.path.exists(dst_lbl):
                open(dst_lbl, 'a').close()
        
        added_count += 1
        
    print(f"\n✅ Done! Added {added_count} items.")
    print(f"Total images now in ir_dataset: {len(os.listdir(DEST_IMG_DIR))}")
else:
    print(f"❌ Error: Could not find {SOURCE_IMG_DIR}. Check if 'label.ipynb' is actually a folder or part of the path.")

📂 Found 408 images. Adding them with prefix 'f3_'...


100%|██████████| 408/408 [00:00<00:00, 10225.24it/s]


✅ Done! Added 408 items.
Total images now in ir_dataset: 1022
